# Jinke reach data generation

Colab workflow for dry-run estimation, five-request ORS test, full one-range fallback generation, and web-data ZIP export. ORS is never called unless `RUN_ORS=True`.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
ORS_API_KEY = userdata.get("ORS_API_KEY")
print("ORS key loaded from Secrets:", bool(ORS_API_KEY))

In [ ]:
from pathlib import Path
from pipeline.generate import Config, load_stations, fill_cache, build_outputs
BASE = Path("/content/drive/MyDrive/Jinke50min")
CFG = Config(cache_dir=BASE/"ors_cache", web_data_dir=Path("web/public/data"), audit_dir=BASE/"audit_outputs", dry_run=True, test_mode=False)
rows = load_stations(CFG)
print("Production Sheet rows:", len(rows))

In [ ]:
# Dry-run only: estimates requests, validates modern and legacy cache, does not call ORS.
dry_report = fill_cache(rows, CFG, api_key=None)
dry_report

In [ ]:
# Optional five-request live ORS smoke test. Run only after reviewing dry_report and quota.
RUN_ORS = False
if RUN_ORS:
    smoke_cfg = Config(cache_dir=BASE/"ors_cache", web_data_dir=Path("web/public/data"), audit_dir=BASE/"audit_outputs", dry_run=False, max_calls=5, test_mode=False)
    smoke_report = fill_cache(rows, smoke_cfg, api_key=ORS_API_KEY)
    print(smoke_report)
else:
    print("Skipped: set RUN_ORS=True in this cell for a five-request test.")

In [ ]:
# Complete one-range fallback generation. Run only when ready to spend quota.
RUN_ORS = False
if RUN_ORS:
    full_cfg = Config(cache_dir=BASE/"ors_cache", web_data_dir=Path("web/public/data"), audit_dir=BASE/"audit_outputs", dry_run=False, max_calls=450, test_mode=False)
    full_report = fill_cache(rows, full_cfg, api_key=ORS_API_KEY)
    manifest = build_outputs(rows, full_cfg)
    print(full_report)
    print(manifest)
else:
    print("Skipped: set RUN_ORS=True in this cell for full generation.")

In [ ]:
# Generate/update static files and web-data.zip from currently available cache/sample fallback.
manifest = build_outputs(rows, CFG)
print("ZIP:", CFG.audit_dir/"web-data.zip")
manifest